# Reproducing Yang et al. (2025) from dandiset 001212

**Paper.** Yang, Ni, Sun, Xing, Wang, Zhang, Zhu and Xu (2025), "Individualized Target Selection of
Closed-Loop Electrical Stimulation for the Treatment of Spontaneous Temporal Lobe Epilepsy", IEEE
Trans. Neural Syst. Rehabil. Eng. 33:4008-4016, doi:10.1109/tnsre.2025.3614867.

**Dataset.** DANDI:001212, version `0.241023.0931` (doi:10.48324/dandi.001212/0.241023.0931),
streamed from the archive. The paper cites no version of the dandiset. This is the only published
version, and it is the most recent; the draft has the same 56 assets and size, so no other version
could match the claims file's description better.

**Claims.** `claims/yang-2025.yaml`, written from the version of record and its supplement (Figures
S1 to S3, Table S1), checked by a second reader. The claims file decides what is tested; method
details it leaves open are taken from the paper, cited where used, and the readings it names are
all run.

**What the dandiset holds.** For each of the 7 rats, 8 NWB files (4 baseline and 4 stimulation
groups); each file holds one 600 s, 4-channel (DG, CA1, CA3, ANT) LFP epoch per seizure at 1 kHz.
There are no seizure times, no stimulation times, no interictal recordings and no recording
durations. Seizure onset and end and the stimulation trains are therefore found from the LFP here
(an assumption, stated below), and the claims that need interictal data or recording time are
blocked.

| Claim | Tested or blocked |
|---|---|
| tables1-network-stable-short-term | tested |
| fig2-r1-target-shifts | tested |
| figs2-target-assignment | tested |
| fig3a-target-reduces-each-rat | tested |
| fig3a-nontarget-can-lengthen | tested |
| fig3a-per-rat-site-differences | tested |
| fig3a-target-vs-nontarget-total | tested |
| fig3a-some-nontarget-shorten | tested |
| fig3b-same-site-target-vs-nontarget | tested |
| seizure-duration-proportion | blocked: data missing (no recording time) |
| fig3c-frequency-no-difference | blocked: data missing (no recording time) |
| fig4b-psd-target-bands | tested |
| fig4b-psd-other-bands-unchanged | tested |
| fig5b-iis-post-seizure | tested |
| fig5c-interictal-iis-no-difference | blocked: data missing (no interictal segments) |
| fig5d-iis-seizure-correlation | blocked: data missing (no interictal segments, no recording time) |
| fig6cd-interictal-hfo-no-difference | blocked: data missing (no interictal segments) |
| fig6e-ictal-ripples-no-difference | tested |
| fig6f-ictal-fast-ripples-reduced | tested |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time
import itertools
import resource
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import yang

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
CH = yang.CHANNELS
print("dandiset", yang.DANDISET, "version", yang.VERSION)

## Loading

Every epoch of every file is streamed once, and everything the claims need is computed from it as
it is read (`yang.epoch_features`): seizure onset and end, stimulation trains, cf_w at onset, PSD
ratios, interictal spike counts around the seizure and ictal HFO rates. Four files are read at a
time, each in its own process.

In [ ]:
assets = yang.list_assets()
t0 = time.time()
rows = yang.process_all(assets, workers=4)
print(f"{len(assets)} files, {len(rows)} epochs, {time.time() - t0:.0f} s; peak memory of a worker "
      f"{resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6:.2f} GB")
df = pd.DataFrame(rows)
df["ntrains"] = df.trains.apply(len)
for th in yang.SZ_THRESH_VARIANTS:
    df[f"dur_{th}"] = df[f"sz_{th}"].apply(lambda x: x[1] - x[0])
df["dur_notrainend"] = df["sz_notrainend"].apply(lambda x: x[1] - x[0])
df["dur"] = df[f"dur_{yang.SZ_THRESH}"]

## Data check

In [ ]:
ad = pd.DataFrame(assets)
print("files per rat and kind:")
print(ad.groupby(["rat", "kind"]).size().unstack())
print("\nepochs per rat and kind:")
print(df.groupby(["rat", "kind"]).size().unstack())
print("\nepoch shapes:", df.groupby(["n_samples", "n_channels"]).size().to_dict(),
      "| rate (Hz):", df.rate.unique(), "| starting_time:", df.start.unique(), "| unit:", df.unit.unique())
print("LFP range over all epochs and channels (microvolts): min %.0f, max %.0f; per-epoch SD median %.1f "
      "(range %.1f to %.1f)" % (np.min(np.vstack(df["min"])), np.max(np.vstack(df["max"])),
                                np.median(np.vstack(df["std"])), np.min(np.vstack(df["std"])),
                                np.max(np.vstack(df["std"]))))
amax = np.maximum(np.abs(np.vstack(df["min"])), np.abs(np.vstack(df["max"]))).max(1)
print(f"epochs reaching |LFP| = {amax.max():.0f} microvolts (the largest value, likely the amplifier's "
      f"limit): {int((amax >= amax.max() - 1e-6).sum())} of {len(df)}; samples at that value in those epochs: "
      f"{int(df.n_at_extreme[amax >= amax.max() - 1e-6].sum())}")
print("each series' description names its own file's group:",
      all(r.description.split(" at ")[1].split(" ")[0].replace("_", "-") ==
          ad.set_index("group").loc[r.group, "path"].split(f"ses-Rat{r.rat}-")[1].split("_ecephys")[0]
          for r in df.itertuples()))

### Duplicates

Each epoch's data, and each channel separately, is hashed (MD5 of the float64 values).

In [ ]:
dup = df.md5.value_counts()
print("epochs whose full data occur more than once:", int((dup > 1).sum()))
chash = pd.Series([h for hs in df.md5_ch for h in hs]).value_counts()
print("channel traces that occur more than once:", int((chash > 1).sum()))
# epochs dated the same day in two groups of one rat
dd = df.groupby(["rat", "date"]).group.nunique()
print("rat-days with epochs in two groups:", dd[dd > 1].index.tolist())

### Group order: file names, epoch dates and Figure S2

The file names number the groups 00 to 07 (baseline, stimulation, alternating). The epoch names
carry dates. Figure S2 gives the order of the stimulated sites.

In [ ]:
FIGS2_ORDER = {  # stimulated site of stimulation groups 1 to 4, Figure S2 (claims file)
    "R1": ["CA1", "CA3", "ANT", "DG"], "R2": ["DG", "ANT", "CA1", "CA3"],
    "R3": ["CA3", "ANT", "DG", "CA1"], "R4": ["CA1", "DG", "ANT", "CA3"],
    "R5": ["ANT", "DG", "CA1", "CA3"], "R6": ["CA1", "ANT", "DG", "CA3"],
    "R7": ["DG", "ANT", "CA1", "CA3"]}
FIGS2_TARGETS = {  # computed target(s) of baseline groups 0 to 3, Figure S2
    "R1": [{"ANT"}, {"ANT"}, {"CA3"}, {"DG"}], "R2": [{"DG", "CA1"}, {"CA1"}, {"CA1"}, {"ANT"}],
    "R3": [{"CA3"}, {"CA3"}, {"ANT"}, {"CA3", "ANT"}], "R4": [{"CA3"}] * 4,
    "R5": [{"CA3", "ANT"}, {"ANT"}, {"ANT"}, {"ANT"}],
    "R6": [{"ANT"}, {"CA1", "CA3"}, {"CA3"}, {"CA3"}], "R7": [{"ANT"}] * 4}
TARGET = {("R1", "DG"), ("R2", "DG"), ("R2", "CA1"), ("R3", "CA3"), ("R4", "CA3"), ("R5", "ANT"),
          ("R6", "CA3"), ("R7", "ANT")}  # target stimulation groups, Figure S2 / Figure 3A boxes
span = df.groupby("group").agg(rat=("rat", "first"), kind=("kind", "first"), site=("site", "first"),
                               n=("epoch", "size"), first=("date", "min"), last=("date", "max"),
                               days=("date", "nunique"))
print(span.to_string())
for rat in sorted(FIGS2_ORDER):
    ds = ad[(ad.rat == rat) & (ad.kind == "stim")].sort_values("index").site.tolist()
    print(rat, "dandiset order", ds, "| Figure S2 order", FIGS2_ORDER[rat],
          "| same" if ds == FIGS2_ORDER[rat] else "| DIFFERENT")

### Stimulation trains and the clock check

The files carry no stimulation times. Trains of 130 Hz pulses appear as a comb of aliased lines in
the 1 kHz LFP; they are detected per 0.5 s window (`yang.find_trains`, at least 4 s long). Methods
C gives 5 s trains with 5 s pauses, which the detected trains are checked against. As the LFP is
the only data stream, the clock check asks whether the trains, which the closed-loop detector
triggers at seizures, fall inside the seizures found from the LFP, against the same with the
seizure times shifted.

In [ ]:
st = df[df.kind == "stim"]
print("stimulation groups: epochs with at least one train, of all epochs")
print(st.groupby("group").agg(site=("site", "first"), epochs=("epoch", "size"),
                              with_trains=("ntrains", lambda x: int((x > 0).sum())),
                              ).T.to_string())
print("pulse rate (Hz) per group, median over epochs with trains:",
      st[st.ntrains > 0].groupby("group").pulse_rate.median().to_dict())
lens = np.array([e - s for tr in st.trains for s, e in tr])
print(f"\ntrain length: n = {len(lens)}, median {np.median(lens):.2f} s, 5th to 95th percentile "
      f"{np.percentile(lens, 5):.2f} to {np.percentile(lens, 95):.2f} s")
gaps = np.array([tr[i + 1][0] - tr[i][1] for tr in st.trains for i in range(len(tr) - 1)])
print(f"pause between consecutive trains: median {np.median(gaps):.2f} s, 25th to 75th percentile "
      f"{np.percentile(gaps, 25):.2f} to {np.percentile(gaps, 75):.2f} s")
bt = df[(df.kind == "baseline") & (df.ntrains > 0)]
print(f"\nbaseline epochs with detected trains: {len(bt)} of {int((df.kind == 'baseline').sum())}")
print(bt[["group", "epoch", "pulse_rate", "trains"]].to_string())


def frac_in(shift):
    k = n = 0
    for r in st[(st.ntrains > 0) & st.onset.notna()].itertuples():
        for s, e in r.trains:
            n += 1
            k += (s < r.end + shift + 1) and (e > r.onset + shift - 1)
    return k / n


print("\nfraction of trains overlapping the seizure found from the LFP, by shift of the seizure times:")
print({sh: round(frac_in(sh), 3) for sh in (-120, -60, -30, 0, 30, 60, 120)})
lat = st[(st.ntrains > 0) & st.onset.notna()].apply(lambda r: r.trains[0][0] - r.onset, axis=1)
print(f"first train minus seizure onset: median {lat.median():.1f} s, 25th to 75th percentile "
      f"{lat.quantile(.25):.1f} to {lat.quantile(.75):.1f} s, range {lat.min():.1f} to {lat.max():.1f}")

### Seizure onset and end

The dandiset gives no seizure times, and the paper gives no rule for marking them (claims file,
opening comment, reading (b)). The assumption here (`yang.detect_seizure`): the largest over
channels of the 1-45 Hz line length in 1 s windows, divided by the channel's median over the epoch,
above 2 (with stimulation trains counted as ictal, runs merged across gaps under 7 s, and
boundaries extended while it stays above 1.5). The thresholds 1.5 and 3, and ending a seizure at
the last ictal window before a final train, are shown as variants in every duration claim.

In [ ]:
for th in yang.SZ_THRESH_VARIANTS:
    c = f"dur_{th}"
    print(f"threshold {th}: seizures found in {int(df[c].notna().sum())} of {len(df)} epochs; duration "
          f"median {df[c].median():.1f} s, range {df[c].min():.1f} to {df[c].max():.1f} s")
print("epochs without a seizure at the main threshold:", df[df.dur.isna()][["group", "epoch"]].values.tolist())
print("\nmedian duration (s) by rat and kind, main threshold:")
print(df.groupby(["rat", "kind"]).dur.median().unstack().round(1))
onsets = df.onset.dropna()
print(f"\nonset within the 600 s epoch: median {onsets.median():.1f} s, range {onsets.min():.1f} to "
      f"{onsets.max():.1f} s")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
axs[0].hist(df.dur.dropna(), bins=40)
axs[0].set_xlabel("seizure duration (s), main threshold")
axs[0].set_ylabel("epochs")
axs[1].hist(lat, bins=30)
axs[1].set_xlabel("first train minus onset (s)")
plt.tight_layout()
plt.show()
print(f"all {int(df.dur.notna().sum())} durations ({df.dur.min():.1f} to {df.dur.max():.1f} s) and "
      f"all {len(lat)} latencies ({lat.min():.1f} to {lat.max():.1f} s) are inside the axes")

## Network claims

cf_w over the first 5 s after onset, 9 windows of 1 s with 0.5 s overlap (Methods B), from
conditional GC with BIC model order at 1 kHz (main) and, as a variant, after decimation to 250 Hz.
Reading (a) of the claims file's sign (out-strength minus in-strength) is `cfw`; reading (b) is its
negative. The two readings of combining windows coincide (cf_w is linear in the GC matrix).

In [ ]:
base = df[(df.kind == "baseline") & df.cfw.notna()].copy()
print("baseline seizures with cf_w:", len(base), "of", int((df.kind == "baseline").sum()))
print("VAR order (BIC, 1 kHz): median", base.gc_order.median(), "range", base.gc_order.min(), "to",
      base.gc_order.max(), "| at 250 Hz: median", base.gc_order_ds4.median())
GC_VARIANTS = {"1 kHz": "cfw", "250 Hz": "cfw_ds4", "1 kHz, onset threshold 1.5": "cfw_1.5",
               "1 kHz, onset threshold 3": "cfw_3.0"}


def pair_sets(V, grp, twice=False, self_pairs=False):
    R = np.corrcoef(V)
    n = len(V)
    same = grp[:, None] == grp[None, :]
    if twice:
        m = ~np.eye(n, dtype=bool)
    else:
        m = np.triu(np.ones((n, n), bool), 1)
    intra = R[m & same]
    inter = R[m & ~same]
    if self_pairs:
        intra = np.concatenate([intra, np.ones(n)])
    return intra, inter

### tables1-network-stable-short-term

> In every rat the cf_w vectors of seizure onset correlate more strongly between seizures of the
> same baseline group than between seizures of different baseline groups (Pearson r, intra-group
> against inter-group, Mann-Whitney).

Criterion: reproduced when, under one reading, intra > inter with Mann-Whitney p < 0.05 in all 7
rats and at least 10 of the 14 means lie within tolerance (Table S1 SDs); partially reproduced when
direction and significance hold in at least 5 rats. Readings: pairs once (a) or twice (b); self
pairs excluded (c) or included (d); and the GC variants.

In [ ]:
TABLE_S1 = {"R1": (.72, .30, -.11, .55), "R2": (.47, .57, .09, .56), "R3": (.77, .19, .59, .34),
            "R4": (.81, .46, .79, .46), "R5": (.74, .34, .71, .28), "R6": (.61, .34, .36, .47),
            "R7": (.81, .36, .71, .41)}
res = []
for gname, col in GC_VARIANTS.items():
    for twice, selfp in itertools.product((False, True), (False, True)):
        for rat, g in base[base[col].notna()].groupby("rat"):
            V = np.vstack(g[col].values)
            grp = np.array(g.group.tolist(), dtype=object)
            intra, inter = pair_sets(V, grp, twice, selfp)
            p = stats.mannwhitneyu(intra, inter, alternative="two-sided").pvalue
            e = TABLE_S1[rat]
            res.append(dict(gc=gname, pairs="twice" if twice else "once", self="incl" if selfp else "excl",
                            rat=rat, n=len(V), intra=intra.mean(), intra_sd=intra.std(),
                            inter=inter.mean(), inter_sd=inter.std(), p=p,
                            ok=(intra.mean() > inter.mean()) and p < 0.05,
                            in_tol=int(abs(intra.mean() - e[0]) <= e[1]) + int(abs(inter.mean() - e[2]) <= e[3])))
t1 = pd.DataFrame(res)
for selfp in ("excl", "incl"):
    print(f"1 kHz, pairs once, self pairs {selfp}:")
    print(t1[(t1.gc == "1 kHz") & (t1.pairs == "once") & (t1.self == selfp)].round(3).to_string())
summ = t1.groupby(["gc", "pairs", "self"]).agg(rats_significant=("ok", "sum"), means_in_tol=("in_tol", "sum"))
print("\nby reading (7 rats, 14 means):")
print(summ.to_string())
print("\nrats significant under each reading, listed:")
print(t1[t1.ok].groupby(["gc", "pairs", "self"]).rat.apply(lambda x: ",".join(x)).to_string())

**Result: reproduced, under reading (d) of the criterion (self pairs included).** With 1 kHz GC
and each pair once, intra-group r exceeds inter-group r with Mann-Whitney p < 0.05 in all 7 rats
and 13 of 14 means are within the Table S1 SDs. The reading matters: with self pairs excluded the
test holds in 4 rats at the main onset (R1, R2, R3, R6; 11 of 14 means within tolerance), and in 6
rats with the onset thresholds 1.5 or 3. Reading (d) adds an r of 1 per seizure to the intra-group
set. The outcome also rests on the seizure onsets found from the LFP (an assumption).

### fig2-r1-target-shifts

> In rat R1 the node with the highest cf_w at seizure onset changes across the four baseline groups
> (ANT, ANT, CA3, DG), while the cf_w pattern is consistent within each group.

Criterion: reproduced when, under one reading, the node with the highest mean cf_w in each of R1's
four baseline groups is the one named; partially when three of the four match.

In [ ]:
r1 = base[base.rat == "R1"]
named = ["ANT", "ANT", "CA3", "DG"]
for gname, col in GC_VARIANTS.items():
    for sign, lab in ((1, "out - in"), (-1, "in - out")):
        tops, means = [], []
        for gg, g in r1[r1[col].notna()].groupby("group"):
            m = sign * np.vstack(g[col].values).mean(0)
            tops.append(CH[int(np.argmax(m))])
            means.append(np.round(m, 3))
        print(f"{gname:28s} {lab}: top nodes {tops}, {sum(a == b for a, b in zip(tops, named))} of 4 match")
print("\nmean cf_w (out - in, 1 kHz) per R1 baseline group, nodes", CH)
for gg, g in r1.groupby("group"):
    V = np.vstack(g.cfw.values)
    print(gg, "n =", len(V), np.round(V.mean(0), 4), "| seizures whose top node is the group's top:",
          int((np.argmax(V, 1) == np.argmax(V.mean(0))).sum()))
fig, ax = plt.subplots(figsize=(10, 2.5))
V = np.vstack(r1.cfw.values).T
im = ax.imshow(V, aspect="auto", cmap="RdBu_r", vmin=-np.abs(V).max(), vmax=np.abs(V).max())
ax.set_yticks(range(4), CH)
b = np.cumsum(r1.groupby("group").size().values)[:-1]
for x in b:
    ax.axvline(x - .5, color="k")
plt.colorbar(im, label="cf_w (out - in)")
ax.set_xlabel("R1 baseline seizures, groups 0 to 3")
plt.show()
print(f"cf_w range shown: {V.min():.3f} to {V.max():.3f}")

**Result: reproduced.** With cf_w as out-strength minus in-strength (1 kHz GC), the top nodes of
R1's four baseline groups are ANT, ANT, CA3, DG, as named. The top node of single seizures agrees
with its group's top in 9 of 9, 8 of 10, 4 of 5 and 6 of 11 seizures. At 250 Hz three of four
match, with onset threshold 3 four, with threshold 1.5 two; with the sign as worded (in minus out)
none. The outcome rests on the onset assumption.

### figs2-target-assignment

> Across the 7 rats, the targets computed from each baseline group's seizures, compared with the
> site stimulated in the next group, give 8 target and 20 non-target stimulation groups as Figure
> S2 shows.

Criterion: reproduced when, under one reading, at least 24 of the 28 baseline groups give the
Figure S2 target set and the labels of all 28 stimulation groups agree with Figure S2; partially
when at least 18 baseline groups match. Readings of "significantly high": (a) the node with the
largest mean cf_w; (b) nodes whose cf_w is significantly above that of each other node
(Kruskal-Wallis, then pairwise Mann-Whitney, p < 0.05). The next group is the next file of the
dandiset.

In [ ]:
def targets_b(V, alpha=0.05):
    if stats.kruskal(*V.T).pvalue >= alpha:
        return set()
    out = set()
    for i in range(4):
        if all(V[:, i].mean() > V[:, j].mean() and
               stats.mannwhitneyu(V[:, i], V[:, j], alternative="greater").pvalue < alpha
               for j in range(4) if j != i):
            out.add(CH[i])
    return out


rows_s2 = []
for gname, col in GC_VARIANTS.items():
    for sign, lab in ((1, "out-in"), (-1, "in-out")):
        for rat in sorted(FIGS2_TARGETS):
            for bi in range(4):
                g = base[(base.rat == rat) & (base["index"] == 2 * bi) & base[col].notna()]
                V = sign * np.vstack(g[col].values)
                ta = {CH[int(np.argmax(V.mean(0)))]}
                tb = targets_b(V)
                fs2 = FIGS2_TARGETS[rat][bi]
                nxt = ad[(ad.rat == rat) & (ad["index"] == 2 * bi + 1)].site.iloc[0]
                rows_s2.append(dict(gc=gname, sign=lab, rat=rat, baseline=bi, figS2=",".join(sorted(fs2)),
                                    top=",".join(ta), sig=",".join(sorted(tb)),
                                    match_a=bool(ta <= fs2), match_b=(tb == fs2), next_site=nxt,
                                    label_a=nxt in ta, label_b=nxt in tb,
                                    label_figS2=(rat, nxt) in TARGET))
s2 = pd.DataFrame(rows_s2)
main = s2[(s2.gc == "1 kHz") & (s2.sign == "out-in")]
print(main.drop(columns=["gc", "sign"]).to_string())
print("\nby reading: baseline groups matching Figure S2 (of 28), stimulation groups labelled target "
      "(Figure S2 gives 8), labels agreeing with Figure S2 (of 28)")
print(s2.groupby(["gc", "sign"]).apply(lambda x: pd.Series(dict(
    match_a=int(x.match_a.sum()), match_b=int(x.match_b.sum()),
    targets_a=int(x.label_a.sum()), targets_b=int(x.label_b.sum()),
    labels_agree_a=int((x.label_a == x.label_figS2).sum()),
    labels_agree_b=int((x.label_b == x.label_figS2).sum())))).to_string())
print("\nFigure S2 labels applied to the dandiset's group order give", sum((r, s) in TARGET for r, s in
      ad[ad.kind == "stim"][["rat", "site"]].values), "target groups by site; Figure S2's targets applied "
      "to the dandiset's order (target when the next file's site is in the preceding baseline's "
      "Figure S2 set):", int(main.apply(lambda r: r.next_site in set(r.figS2.split(",")), axis=1).sum()))

**Result: partially reproduced.** Under reading (a), the top node, with 1 kHz GC and cf_w as
out minus in, 18 of 28 baseline groups give the Figure S2 target set (R1, R3, R4 and R7 all four;
R2 one; R5 none; R6 one), and the labels of 20 of 28 stimulation groups agree with Figure S2 (8
groups are labelled target, as in the paper, but not the same 8). At 250 Hz 18 groups match and
24 labels agree. Reading (b), significance, matches 11 or 12 groups at most. The dandiset's order
of stimulated sites differs from Figure S2's for R5 and R6.

## Seizure duration claims

Normalized duration of each seizure of a stimulation group against the preceding baseline group
(the previous file of the dandiset): (a) relative change, duration / baseline mean - 1 (main);
(b) difference from the baseline mean in s; (c) z-score against the baseline seizures. Target and
non-target labels are those of Figure S2 by rat and site, as the claims file says. Seizure sets: (a)
all seizures of the group (main); (b) only those with a detected stimulation train. Detector
variants: thresholds 1.5 and 3, and ending at the last ictal window before a final train. A further
variant takes the preceding baseline from Figure S2's order, which differs from the dandiset's for
R5 and R6.

In [ ]:
def norm_table(dcol="dur", seizures="all", base_order="dandiset"):
    out = []
    for r in ad[ad.kind == "stim"].itertuples():
        g = df[(df.group == r.group)]
        if seizures == "stimulated":
            g = g[g.ntrains > 0]
        d = g[dcol].dropna().values
        bi = r.index - 1
        if base_order == "figS2":
            bi = 2 * FIGS2_ORDER[r.rat].index(r.site)
        b = df[(df.rat == r.rat) & (df["index"] == bi)][dcol].dropna().values
        out.append(dict(rat=r.rat, site=r.site, group=r.group, target=(r.rat, r.site) in TARGET,
                        n=len(d), base_n=len(b), base_mean=b.mean(),
                        rel=d / b.mean() - 1, diff=d - b.mean(), z=(d - b.mean()) / b.std(ddof=1),
                        d=d, b=b))
    t = pd.DataFrame(out)
    for k in ("rel", "diff", "z"):
        t[k + "_mean"] = t[k].apply(lambda x: np.mean(x) if len(x) else np.nan)
    return t


VARIANTS = {"main": dict(), "threshold 1.5": dict(dcol="dur_1.5"), "threshold 3": dict(dcol="dur_3.0"),
            "end before final train": dict(dcol="dur_notrainend"),
            "stimulated seizures only": dict(seizures="stimulated"),
            "baseline in Figure S2 order": dict(base_order="figS2")}
NT = {k: norm_table(**v) for k, v in VARIANTS.items()}
print(NT["main"][["group", "site", "target", "n", "base_n", "base_mean", "rel_mean", "diff_mean", "z_mean"]]
      .round(3).to_string())

### fig3a-target-reduces-each-rat

> In each of the 7 rats, stimulation of the target site reduced normalized seizure duration below
> the preceding baseline.

Criterion: reproduced when, under one reading, the mean normalized duration is below zero in all 8
target groups; partially in 6 or 7. The three normalizations have the same sign by construction,
so the readings that can differ are the seizure set and the detector.

In [ ]:
FIG3A = {("R1", "DG"): -.10, ("R2", "DG"): -.37, ("R2", "CA1"): -.09, ("R3", "CA3"): -.23,
         ("R4", "CA3"): -.14, ("R5", "ANT"): -.33, ("R6", "CA3"): -.21, ("R7", "ANT"): -.13}
tab = []
for k, t in NT.items():
    tt = t[t.target].set_index(["rat", "site"])
    row = {f"{a} {b}": round(tt.loc[(a, b), "rel_mean"], 3) for a, b in FIG3A}
    row["below zero (of 8)"] = int((tt.rel_mean < 0).sum())
    tab.append(pd.Series(row, name=k))
print(pd.DataFrame(tab).T.to_string())
print("\nFigure 3A, read from the bars (not judged):", {f"{a} {b}": v for (a, b), v in FIG3A.items()})

**Result: not reproduced.** The mean normalized duration is below zero in 3 of the 8 target
groups (R3 CA3 -0.03, R6 CA3 -0.18, R7 ANT -0.21) with the main detector, and in at most 4 of 8
under any variant (thresholds 1.5 and 3, ending before a final train, stimulated seizures only,
baseline in Figure S2 order). The durations rest on the onset and end assumption.

### fig3a-nontarget-can-lengthen

> Stimulation of some non-target sites increased seizure duration above the preceding baseline.

Criterion: reproduced when, under one reading, at least 5 of the 20 non-target groups have a positive
mean normalized duration; not reproduced when fewer than 2 do.

In [ ]:
for k, t in NT.items():
    nt = t[~t.target]
    pos = nt[nt.rel_mean > 0]
    print(f"{k:28s}: {len(pos)} of {nt.rel_mean.notna().sum()} non-target groups positive: "
          + ", ".join(f"{r.rat} {r.site} ({r.rel_mean:+.2f})" for r in pos.itertuples()))
print("Figure 3A shows positive bars for R1 CA3 and ANT, R2 CA3, R3 DG and ANT, R4 DG and ANT, R5 CA1, "
      "R6 DG, R7 DG")

**Result: reproduced.** 13 of the 20 non-target groups have a positive mean normalized
duration with the main detector, and 10 to 14 under the variants; the criterion asks for 5.

### fig3a-per-rat-site-differences

> Within rats R2, R3, R4, R5 and R7 the four stimulation sites differ in normalized seizure duration
> (Kruskal-Wallis), while in R1 and R6 no pair of sites differs.

Criterion: per rat, Kruskal-Wallis across the four stimulation groups, then Dunn's test, (a)
Bonferroni-adjusted or (b) unadjusted. Reproduced when at least 8 of the 11 listed pairs are
significant and R1 and R6 have none; partially when at least 5 are.

In [ ]:
def dunn(samples):
    """Dunn's test after Kruskal-Wallis, with the tie correction; two-sided p per pair."""
    allv = np.concatenate(samples)
    ranks = stats.rankdata(allv)
    N = len(allv)
    _, cnt = np.unique(allv, return_counts=True)
    tie = np.sum(cnt ** 3 - cnt) / (12 * (N - 1))
    idx = np.cumsum([0] + [len(s) for s in samples])
    mr = [ranks[idx[i]:idx[i + 1]].mean() for i in range(len(samples))]
    out = {}
    for i, j in itertools.combinations(range(len(samples)), 2):
        se = np.sqrt((N * (N + 1) / 12 - tie) * (1 / len(samples[i]) + 1 / len(samples[j])))
        out[(i, j)] = 2 * stats.norm.sf(abs(mr[i] - mr[j]) / se)
    return out


LISTED = {("R2", "DG", "CA3"), ("R2", "CA3", "ANT"), ("R3", "DG", "CA1"), ("R3", "DG", "CA3"),
          ("R3", "CA3", "ANT"), ("R4", "DG", "CA1"), ("R4", "DG", "CA3"), ("R5", "CA1", "CA3"),
          ("R5", "CA1", "ANT"), ("R7", "DG", "CA3"), ("R7", "DG", "ANT")}
summary = []
for k, t in NT.items():
    lines = []
    sig_listed = {"bonferroni": 0, "unadjusted": 0}
    other = {"bonferroni": [], "unadjusted": []}
    for rat, g in t.groupby("rat"):
        g = g[g.n > 0]
        samples = [x for x in g.rel]
        sites = g.site.tolist()
        kw = stats.kruskal(*samples).pvalue
        pd_ = dunn(samples)
        for (i, j), p in pd_.items():
            pair = (rat,) + tuple(sorted((sites[i], sites[j]), key=["DG", "CA1", "CA3", "ANT"].index))
            for adj, pp in (("bonferroni", min(1, p * len(pd_))), ("unadjusted", p)):
                if pp < 0.05:
                    if pair in LISTED:
                        sig_listed[adj] += 1
                    else:
                        other[adj].append(f"{pair[0]} {pair[1]}-{pair[2]}")
            if k == "main":
                lines.append(f"{rat} {sites[i]:>3s}-{sites[j]:<3s} Dunn p = {p:.3g} (Bonferroni {min(1, p * 6):.3g})"
                             f"{'  listed' if pair in LISTED else ''}")
        if k == "main":
            lines.append(f"{rat} Kruskal-Wallis p = {kw:.3g}, n = {[len(s) for s in samples]}")
    if k == "main":
        print("\n".join(lines))
    for adj in ("bonferroni", "unadjusted"):
        summary.append(dict(variant=k, posthoc=adj, listed_significant=sig_listed[adj],
                            other_significant=", ".join(other[adj])))
print()
print(pd.DataFrame(summary).to_string())

**Result: partially reproduced.** With the main detector, 5 of the 11 listed pairs are
significant by Dunn's test, adjusted or not (R2 ANT-CA3, R3 DG-CA3, R3 DG-CA1, R4 DG-CA1, R4
DG-CA3); none in R5 and R7 (Kruskal-Wallis p 0.66 and 0.33). R1 has no significant pair, but R6 has
one (DG-ANT, Bonferroni p 0.021), against the claim. Variants give 2 to 6 listed pairs.

### fig3a-target-vs-nontarget-total

> Pooled over rats, target-site stimulation shortened normalized seizure duration (-0.26 +- 0.03)
> significantly more than non-target stimulation (0.038 +- 0.03).

Criterion: reproduced when, under one reading, both means are within tolerance (0.03) and
Mann-Whitney p < 0.001 with target lower; partially when target is lower with p < 0.05 but a mean is
outside tolerance or p is between 0.001 and 0.05. Units: (a) seizures pooled; (b) stimulation groups.

In [ ]:
out = []
for k, t in NT.items():
    for norm in ("rel", "diff", "z"):
        T = np.concatenate(t[t.target][norm].values)
        N = np.concatenate(t[~t.target][norm].values)
        p = stats.mannwhitneyu(T, N, alternative="two-sided").pvalue
        Tg, Ng = t[t.target][norm + "_mean"].dropna(), t[~t.target][norm + "_mean"].dropna()
        pg = stats.mannwhitneyu(Tg, Ng, alternative="two-sided").pvalue
        out.append(dict(variant=k, norm=norm, target=T.mean(), target_sem=stats.sem(T), nontarget=N.mean(),
                        nontarget_sem=stats.sem(N), n_t=len(T), n_n=len(N), p_seizures=p,
                        groups_target=Tg.mean(), groups_nontarget=Ng.mean(), p_groups=pg))
tot = pd.DataFrame(out)
tot["both_in_tol"] = (abs(tot.target + .26) <= .03) & (abs(tot.nontarget - .038) <= .03)
print(tot.round(4).to_string())

**Result: not reproduced.** Pooled over seizures, target -0.020 +- 0.028 and non-target
0.087 +- 0.034 (relative change), Mann-Whitney p = 0.085: target is lower but not significantly,
and the target mean is far outside -0.26 +- 0.03. No variant gives p < 0.05 (smallest 0.055), and
over groups p is 0.10 or more.

### fig3a-some-nontarget-shorten

> Five non-target stimulations (R2 ANT, R3 CA1, R4 CA1, R5 CA3, R7 CA3) also significantly
> shortened seizure duration, but less than the target site of the same rat.

Criterion: readings (a) Mann-Whitney of the group's durations against the preceding baseline's;
(b) Wilcoxon signed-rank of normalized durations against zero; (c) one-sample t-test against zero.
Reproduced when, under one reading, at least 4 of the 5 named groups are significantly shorter and
in at least 4 of the 5 rats the named group's mean is above the target group's (R2: DG target);
partially when 3 of each hold.

In [ ]:
NAMED = [("R2", "ANT"), ("R3", "CA1"), ("R4", "CA1"), ("R5", "CA3"), ("R7", "CA3")]
RAT_TARGET = {"R2": "DG", "R3": "CA3", "R4": "CA3", "R5": "ANT", "R7": "ANT"}
summ = []
for k, t in NT.items():
    ti = t.set_index(["rat", "site"])
    rows_ = []
    for rat, site in NAMED:
        r = ti.loc[(rat, site)]
        if r.n < 2:
            rows_.append(dict(group=f"{rat} {site}", n=r.n))
            continue
        pa = stats.mannwhitneyu(r.d, r.b, alternative="less").pvalue
        pb = stats.wilcoxon(r.rel, alternative="less").pvalue
        pc = stats.ttest_1samp(r.rel, 0, alternative="less").pvalue
        tm = ti.loc[(rat, RAT_TARGET[rat]), "rel_mean"]
        rows_.append(dict(group=f"{rat} {site}", n=r.n, mean=r.rel_mean, p_a=pa, p_b=pb, p_c=pc,
                          target_mean=tm, less_effective=r.rel_mean > tm))
    x = pd.DataFrame(rows_)
    if k == "main":
        print(x.round(4).to_string())
    summ.append(dict(variant=k, sig_a=int((x.p_a < .05).sum()), sig_b=int((x.p_b < .05).sum()),
                     sig_c=int((x.p_c < .05).sum()), less_effective_than_target=int(x.less_effective.sum())))
print()
print(pd.DataFrame(summ).to_string())
others = []
t = NT["main"]
for r in t[~t.target].itertuples():
    if (r.rat, r.site) not in NAMED and r.n >= 2:
        if stats.mannwhitneyu(r.d, r.b, alternative="less").pvalue < .05:
            others.append(f"{r.rat} {r.site}")
print("other non-target groups significantly shorter than baseline by reading (a), main:", others)

**Result: not reproduced.** With the main detector, 2 of the 5 named groups are significantly
shorter than baseline by readings (a) and (b) (R2 ANT, R7 CA3 by (a); R2 ANT, R3 CA1 by (b)), and 3
by (c); the named group's mean is above the target group's in 2 of 5 rats (R4, R5). No variant
reaches 3 and 3.

### fig3b-same-site-target-vs-nontarget

> For the same site, stimulation as a target shortened normalized seizure duration more than
> stimulation of that site as a non-target, significantly for DG, CA3 and ANT but not CA1.

Criterion: per site, per-seizure normalized durations of target groups against non-target groups
pooled over rats, Mann-Whitney (unpaired t-test as a variant). Reproduced when, under one
normalization reading, DG, CA3 and ANT each show target lower with p < 0.05; partially for two.

In [ ]:
out = []
for k, t in NT.items():
    for norm in ("rel", "diff", "z"):
        for site in ("DG", "CA1", "CA3", "ANT"):
            g = t[t.site == site]
            T = np.concatenate(g[g.target][norm].values)
            N = np.concatenate(g[~g.target][norm].values)
            out.append(dict(variant=k, norm=norm, site=site, target=T.mean(), nontarget=N.mean(),
                            n_t=len(T), n_n=len(N),
                            p_mw=stats.mannwhitneyu(T, N, alternative="two-sided").pvalue,
                            p_t=stats.ttest_ind(T, N).pvalue))
f3b = pd.DataFrame(out)
f3b["target_lower_sig"] = (f3b.target < f3b.nontarget) & (f3b.p_mw < 0.05)
print(f3b[f3b.variant == "main"].round(4).to_string())
print()
print(f3b.groupby(["variant", "norm"]).apply(
    lambda x: ",".join(x[x.target_lower_sig & x.site.isin(["DG", "CA3", "ANT"])].site)).to_string())

**Result: not reproduced.** Only ANT shows target lower with p < 0.05 (relative change, main:
-0.11 against 0.12, p = 0.025); DG (p = 0.51) and CA3 (p = 0.12) do not, and in DG the target
mean is higher. Other normalizations and variants give at most one of the three sites.

## PSD claims

PSD ratio per band and channel in the pre and post windows of Methods D (stimulation groups: onset
to the first train, then the 5 s after the first train; baseline: 0-5 s and 10-15 s after onset).
Stimulation seizures without a train after onset have no window and are left out. Conditions:
sham (baseline seizures), target and non-target. Readings: change measure (a) (post - pre) / pre,
(b) log2(post / pre), (c) post - pre; Welch segments 1 s or 0.5 s; total range 0.5-120 Hz or
0-500 Hz; channels all four (a), or the stimulated channel only for stimulation groups (b);
post hoc Dunn Bonferroni or unadjusted. Unit: seizure-channel pairs.

In [ ]:
BANDS = list(yang.BANDS)
psd = df[df.psd_pre_1000.notna()].copy()
psd["cond"] = psd.apply(lambda r: "sham" if r.kind == "baseline" else
                        ("target" if (r.rat, r.site) in TARGET else "nontarget"), axis=1)
print("seizures with PSD windows:", psd.groupby("cond").size().to_dict())
print("by rat:", psd.groupby(["rat", "cond"]).size().unstack().fillna(0).astype(int).to_dict())


def psd_change(band, measure, nps, total, channels):
    key_pre = f"psd_pre_{nps}" + ("_total500" if total == "0-500" else "")
    key_post = f"psd_post_{nps}" + ("_total500" if total == "0-500" else "")
    out = {}
    for c, g in psd.groupby("cond"):
        vals = []
        for r in g.itertuples():
            pre, post = getattr(r, key_pre)[band], getattr(r, key_post)[band]
            chs = range(4) if (channels == "all" or c == "sham") else [CH.index(r.site)]
            for ch in chs:
                a, b = pre[ch], post[ch]
                vals.append((b - a) / a if measure == "rel" else np.log2(b / a) if measure == "log2" else b - a)
        out[c] = np.array(vals)
    return out


READ = list(itertools.product(("rel", "log2", "diff"), (1000, 500), ("0.5-120", "0-500"), ("all", "stimulated")))
READ = [r for r in READ if not (r[1] == 500 and r[2] == "0-500")]  # 0-500 Hz total computed with 1 s segments
prow = []
for measure, nps, total, chans in READ:
    for band in BANDS:
        x = psd_change(band, measure, nps, total, chans)
        S, Tt, Nn = x["sham"], x["target"], x["nontarget"]
        kw = stats.kruskal(S, Tt, Nn).pvalue
        dn = dunn([Tt, S, Nn])
        prow.append(dict(measure=measure, nperseg=nps, total=total, channels=chans, band=band,
                         sham=np.mean(S), target=np.mean(Tt), nontarget=np.mean(Nn),
                         sham_med=np.median(S), target_med=np.median(Tt), nontarget_med=np.median(Nn),
                         kw_p=kw, p_t_vs_sham=dn[(0, 1)], p_t_vs_nt=dn[(0, 2)],
                         n=(len(S), len(Tt), len(Nn))))
P = pd.DataFrame(prow)
print(P[(P.measure == "rel") & (P.nperseg == 1000) & (P.total == "0.5-120") & (P.channels == "all")]
      .round(4).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
m = P[(P.measure == "log2") & (P.nperseg == 1000) & (P.total == "0.5-120") & (P.channels == "all")]
for i, c in enumerate(("sham", "target", "nontarget")):
    ax.bar(np.arange(6) + i * .27, m[c], width=.27, label=c)
ax.set_xticks(np.arange(6) + .27, BANDS)
ax.set_ylabel("mean log2(post / pre) PSD ratio")
ax.legend()
plt.show()
print("plotted means range", round(m[["sham", "target", "nontarget"]].min().min(), 3), "to",
      round(m[["sham", "target", "nontarget"]].max().max(), 3))

### fig4b-psd-target-bands

> After target stimulation the change in PSD ratio was larger in low bands (alpha, and theta or
> beta) and more negative in gamma than after sham (baseline) or non-target stimulation.

Criterion: reproduced when, under one reading of the method and one of the bands ((i) alpha and
theta up, gamma down; (ii) alpha and beta up, gamma down), target differs from both sham and
non-target in the named direction with p < 0.05 (Kruskal-Wallis, then the post hoc test) in all
three bands; partially when alpha and gamma hold.

In [ ]:
def holds(r, up, adj):
    k = 3 if adj == "bonferroni" else 1
    if r.kw_p >= 0.05:
        return False
    if up:
        d = r.target > r.sham and r.target > r.nontarget
    else:
        d = r.target < r.sham and r.target < r.nontarget
    # direction judged on medians as well as means, as the test is rank based
    if up:
        d = d and r.target_med > r.sham_med and r.target_med > r.nontarget_med
    else:
        d = d and r.target_med < r.sham_med and r.target_med < r.nontarget_med
    return d and min(1, r.p_t_vs_sham * k) < 0.05 and min(1, r.p_t_vs_nt * k) < 0.05


ev = []
for (measure, nps, total, chans), g in P.groupby(["measure", "nperseg", "total", "channels"]):
    gb = g.set_index("band")
    for adj in ("bonferroni", "unadjusted"):
        h = {b: holds(gb.loc[b], b != "gamma", adj) for b in ("theta", "alpha", "beta", "gamma")}
        ev.append(dict(measure=measure, nperseg=nps, total=total, channels=chans, posthoc=adj, **h,
                       reading_i=h["theta"] and h["alpha"] and h["gamma"],
                       reading_ii=h["beta"] and h["alpha"] and h["gamma"],
                       alpha_and_gamma=h["alpha"] and h["gamma"]))
E = pd.DataFrame(ev)
print(E.to_string())
print("\nreadings where all three bands hold:", int((E.reading_i | E.reading_ii).sum()), "of", len(E),
      "| alpha and gamma hold:", int(E.alpha_and_gamma.sum()))
print("\nalpha and gamma per reading (KW p, target vs sham p, target vs non-target p, Dunn unadjusted):")
print(P[P.band.isin(["alpha", "gamma", "theta", "beta"])][["measure", "nperseg", "total", "channels", "band",
      "sham_med", "target_med", "nontarget_med", "kw_p", "p_t_vs_sham", "p_t_vs_nt"]].round(4).to_string())

**Result: not reproduced.** In none of the 36 readings of method and post hoc test do the three bands hold, nor
alpha and gamma together. With relative change, 1 s segments, 0.5-120 Hz and all channels, theta
falls after target stimulation (median -0.45, against -0.03 sham and -0.34 non-target), alpha
does not differ (Kruskal-Wallis p = 0.093), beta rises without significance (p = 0.068), and
gamma differs (p = 0.045) with target below sham (Dunn p = 0.020) but not below non-target
(p = 0.31). The stimulation windows rest on the trains found here; R7 contributes one target
and one non-target seizure.

### fig4b-psd-other-bands-unchanged

> In the other bands (delta and high gamma, and whichever of theta and beta is not named as
> changed) the change in PSD ratio did not differ between conditions.

Criterion: a claim of absence, judged on Kruskal-Wallis across sham, target and non-target at
p < 0.05. Delta and high gamma are judged in every reading; beta under band reading (i) and theta
under (ii). fig4b-psd-target-bands was decided by neither band reading, so both are shown, under the
main method reading ((a) relative change, 1 s segments, 0.5-120 Hz, all channels).

In [ ]:
mm = P[(P.measure == "rel") & (P.nperseg == 1000) & (P.total == "0.5-120") & (P.channels == "all")].set_index("band")
for b in ("delta", "high_gamma", "beta", "theta"):
    r = mm.loc[b]
    print(f"{b:10s} KW p = {r.kw_p:.3g}; mean change sham {r.sham:+.3f}, target {r.target:+.3f}, "
          f"non-target {r.nontarget:+.3f}; medians {r.sham_med:+.3f}, {r.target_med:+.3f}, {r.nontarget_med:+.3f}")
print("\nKW p for delta and high gamma under every method reading:")
print(P[P.band.isin(["delta", "high_gamma"])].pivot_table(index=["measure", "nperseg", "total", "channels"],
      columns="band", values="kw_p").to_string())
# effect size: epsilon squared of the KW test
for b in ("delta", "high_gamma", "beta", "theta"):
    x = psd_change(b, "rel", 1000, "0.5-120", "all")
    H = stats.kruskal(*x.values()).statistic
    n = sum(len(v) for v in x.values())
    print(f"{b}: KW H = {H:.1f}, epsilon squared = {H / (n - 1):.3f}")

**Result: not reproduced.** Delta differs between conditions under every method reading
(main Kruskal-Wallis p = 4.8e-21, epsilon squared 0.053) and high gamma under the main reading
(p = 0.0087, epsilon squared 0.005). Beta, judged under band reading (i), does not differ
(p = 0.068); theta, judged under (ii), does (p = 2.3e-13).

## fig5b-iis-post-seizure

> The IIS rate 15 s to 45 s after seizures was lower after target stimulation than after
> non-target stimulation.

Criterion: IIS detected as in Methods F on all channels in the 180 s before onset and after the
end, rate per second. Baseline for the 5 SD threshold: (a) the 180 s pre-seizure stretch (the
interictal segments of reading (b) are not in the dandiset). Tests: (a) Mann-Whitney of per
seizure-channel mean rates over 15-45 s; (b) paired t-test across time bins of 15-45 s (1 s or
5 s bins), target curve against non-target curve. Reproduced when target is lower with p < 0.001;
partially with p < 0.05. Bins after the end of the 600 s epoch are missing and left out; times
inside a stimulation train are not searched.

In [ ]:
ii = df[df.iis_post.notna()].copy()
ii["cond"] = ii.apply(lambda r: "sham" if r.kind == "baseline" else
                      ("target" if (r.rat, r.site) in TARGET else "nontarget"), axis=1)
curves = {}
win_rate = {}
for c, g in ii.groupby("cond"):
    post = np.vstack(list(g.iis_post))  # seizure-channel x 180 bins of 1 s
    pre = np.vstack(list(g.iis_pre))
    curves[c] = (np.nanmean(pre, 0)[::-1], np.nanmean(post, 0))
    win_rate[c] = np.nanmean(post[:, 15:45], 1)
    print(f"{c:9s}: {len(g)} seizures, {post.shape[0]} seizure-channels; rate 15-45 s after end "
          f"{np.nanmean(win_rate[c]):.3f} /s (SEM {stats.sem(win_rate[c], nan_policy='omit'):.3f}); "
          f"180 s before onset {np.nanmean(pre):.3f} /s; missing bins in 15-45 s: {int(np.isnan(post[:, 15:45]).sum())}")
T, N = win_rate["target"], win_rate["nontarget"]
T, N = T[~np.isnan(T)], N[~np.isnan(N)]
print(f"\n(a) Mann-Whitney target vs non-target, seizure-channels: p = {stats.mannwhitneyu(T, N).pvalue:.3g} "
      f"(target {T.mean():.3f}, non-target {N.mean():.3f} /s, difference {T.mean() - N.mean():+.3f})")
for bw in (1, 5):
    ct = curves["target"][1][15:45].reshape(-1, bw).mean(1)
    cn = curves["nontarget"][1][15:45].reshape(-1, bw).mean(1)
    print(f"(b) paired t-test over {len(ct)} bins of {bw} s: p = {stats.ttest_rel(ct, cn).pvalue:.3g}, "
          f"mean difference {np.mean(ct - cn):+.3f} /s")
# per seizure (mean over channels), and per rat
segs = {c: g.apply(lambda r: np.nanmean(r.iis_post[:, 15:45]), axis=1) for c, g in ii.groupby("cond")}
print(f"per seizure (channel mean): target {segs['target'].mean():.3f}, non-target {segs['nontarget'].mean():.3f}, "
      f"Mann-Whitney p = {stats.mannwhitneyu(segs['target'].dropna(), segs['nontarget'].dropna()).pvalue:.3g}")
ii["w"] = ii.apply(lambda r: np.nanmean(r.iis_post[:, 15:45]), axis=1)
print("\nper rat, mean rate 15-45 s (/s):")
print(ii.groupby(["rat", "cond"]).w.mean().unstack().round(3).to_string())
fig, ax = plt.subplots(figsize=(9, 3))
for c, (pre, post) in curves.items():
    ax.plot(np.arange(-180, 0) + .5, pre, lw=.8, label=c)
    ax.plot(np.arange(0, 180) + .5, post, lw=.8, color=ax.lines[-1].get_color())
ax.axvspan(15, 45, color="0.9")
ax.set_xlabel("s before onset (left) and after end (right), the seizure itself removed")
ax.set_ylabel("IIS rate (/s)")
ax.legend()
plt.show()
print("curve range:", round(min(np.nanmin(v) for p in curves.values() for v in p), 3), "to",
      round(max(np.nanmax(v) for p in curves.values() for v in p), 3))

**Result: reproduced.** The IIS rate 15-45 s after the seizure is 0.142 /s after target and
0.210 /s after non-target stimulation. By reading (b), the paired t-test over 30 bins of 1 s,
p = 2.1e-09; over 6 bins of 5 s p = 0.0024; by reading (a), Mann-Whitney over seizure-channels,
p = 0.0032, which alone would be partial. Target is lower than non-target in 5 of 7 rats (not R3
and R4). Reading (b) treats time bins as independent samples. The outcome rests on the seizure end
found from the LFP and on the pre-seizure stretch as the threshold baseline.

## Ictal HFO claims

Ripple (80-250 Hz) and fast ripple (250 to just below 500 Hz) detections by short line length in
the PSD windows (pre and post the first train; baseline 0-5 s and 10-15 s after onset), the
threshold being mean + 3 SD (main) or 5 SD of the short line length over the 180 s before onset
(main), or over the seizure's own two windows pooled (a variant: with the pre-seizure reference
nearly every ictal window is above threshold).
No manual review (it cannot be repeated). Normalized to pre-stimulation as relative change of the
rate. Pairing: (a) by rat, the target groups' mean against the mean of the non-target groups (and
of the baseline groups); (b) unpaired t-test over groups. Channels: (a) all four; (b) the stimulated
channel (all four for baseline). The group value is the group's mean post rate over its mean pre
rate, minus 1, so that windows with no pre event do not divide by zero.

In [ ]:
hf = df[df["hfo_ripple_3.0"].notna()].copy()
hf["cond"] = hf.apply(lambda r: "baseline" if r.kind == "baseline" else
                      ("target" if (r.rat, r.site) in TARGET else "nontarget"), axis=1)
print("seizures with HFO windows:", hf.groupby("cond").size().to_dict())


def hfo_groups(nm, th, chans, ref=""):
    out = []
    for gname, g in hf.groupby("group"):
        r0 = g.iloc[0]
        chs = list(range(4)) if (chans == "all" or r0.kind == "baseline") else [CH.index(r0.site)]
        pre = np.concatenate([x["pre"][chs] for x in g[f"hfo_{nm}_{th}{ref}"]])
        post = np.concatenate([x["post"][chs] for x in g[f"hfo_{nm}_{th}{ref}"]])
        out.append(dict(group=gname, rat=r0.rat, cond=r0.cond, n=len(g), pre=pre.mean(), post=post.mean(),
                        change=post.mean() / pre.mean() - 1 if pre.mean() > 0 else np.nan))
    return pd.DataFrame(out)


def hfo_tests(nm, th, chans, ref=""):
    G = hfo_groups(nm, th, chans, ref)
    per_rat = G.groupby(["rat", "cond"]).change.mean().unstack()
    res = dict(band=nm, thresh=th, channels=chans, reference="pre-seizure 180 s" if ref == "" else "own windows")
    for c in ("target", "nontarget", "baseline"):
        res[c] = G[G.cond == c].change.mean()
    for other in ("nontarget", "baseline"):
        pr = per_rat[["target", other]].dropna()
        res[f"paired_p_vs_{other}"] = stats.ttest_rel(pr.target, pr[other]).pvalue if len(pr) > 1 else np.nan
        res[f"n_rats_vs_{other}"] = len(pr)
        a, b = G[G.cond == "target"].change.dropna(), G[G.cond == other].change.dropna()
        res[f"unpaired_p_vs_{other}"] = stats.ttest_ind(a, b).pvalue
    return res, G, per_rat


H = pd.DataFrame([hfo_tests(nm, th, ch, ref)[0] for nm in ("ripple", "fr") for th in (3.0, 5.0)
                  for ch in ("all", "stimulated") for ref in ("", "_self")])
print(H.round(4).to_string())

### fig6e-ictal-ripples-no-difference

> During seizures, ripple rates normalized to pre-stimulation did not differ between target and
> non-target stimulation.

Criterion: a claim of absence, the paired t-test at p < 0.05, target against non-target; reproduced
when not significant under pairing (a); not reproduced when it is.

In [ ]:
res, G, per_rat = hfo_tests("ripple", 3.0, "all")
print("main reading (3 SD, all channels), per rat:")
print(per_rat.round(3).to_string())
print(f"\npaired t-test by rat, target vs non-target: p = {res['paired_p_vs_nontarget']:.3g} over "
      f"{res['n_rats_vs_nontarget']} rats; target {res['target']:+.3f}, non-target {res['nontarget']:+.3f} "
      f"(difference {res['target'] - res['nontarget']:+.3f}); unpaired over groups p = {res['unpaired_p_vs_nontarget']:.3g}")
print("groups without a value (no pre event or no window):", G[G.change.isna()].group.tolist())
print("Figure 6E (read from the plot, not judged): target about -0.16, non-target about -0.04")

**Result: reproduced.** The paired t-test by rat does not find a difference in normalized ictal
ripple rate between target and non-target stimulation (main reading p = 0.26, 7 rats; target
+2.29, non-target +3.48). One of the 8 method readings is significant (5 SD, stimulated channel,
pre-seizure reference: p = 0.019, target lower). Ripple rates rise from the pre to the post
window in every condition, unlike Figure 6E.

### fig6f-ictal-fast-ripples-reduced

> During seizures, target stimulation reduced the fast ripple rate, normalized to pre-stimulation,
> compared with baseline and with non-target stimulation.

Criterion: reproduced when, under one pairing reading, the target mean is below both others with
p < 0.05 for each comparison; partially when one comparison holds.

In [ ]:
res, G, per_rat = hfo_tests("fr", 3.0, "all")
print("main reading (3 SD, all channels), per rat:")
print(per_rat.round(3).to_string())
for other in ("baseline", "nontarget"):
    print(f"target vs {other}: target {res['target']:+.3f}, {other} {res[other]:+.3f}; paired by rat p = "
          f"{res[f'paired_p_vs_{other}']:.3g} ({res[f'n_rats_vs_{other}']} rats); unpaired p = "
          f"{res[f'unpaired_p_vs_{other}']:.3g}")
fr = H[H.band == "fr"].copy()
fr["below_both"] = (fr.target < fr.baseline) & (fr.target < fr.nontarget)
print("\nall fast ripple readings:")
print(fr.round(4).to_string())
print("Figure 6F (read from the plot, not judged): target about -0.25, baseline about 0.2, non-target about 0.1")

**Result: not reproduced.** The normalized fast ripple rate after target stimulation (+4.16) is
not below baseline (+2.31) or non-target (+18.67) in the main reading (paired p 0.20 and 0.29),
and the target mean is below both in none of the 8 readings.

## Summary

| Claim | Outcome |
|---|---|
| tables1-network-stable-short-term | reproduced (under the self-pairs reading; rests on the onset assumption) |
| fig2-r1-target-shifts | reproduced (rests on the onset assumption) |
| figs2-target-assignment | partially reproduced (18 of 28 baseline groups) |
| fig3a-target-reduces-each-rat | not reproduced (3 of 8 target groups below baseline) |
| fig3a-nontarget-can-lengthen | reproduced (13 of 20) |
| fig3a-per-rat-site-differences | partially reproduced (5 of 11 pairs) |
| fig3a-target-vs-nontarget-total | not reproduced (-0.020 against 0.087, p = 0.085) |
| fig3a-some-nontarget-shorten | not reproduced |
| fig3b-same-site-target-vs-nontarget | not reproduced (ANT only) |
| seizure-duration-proportion | blocked: data missing |
| fig3c-frequency-no-difference | blocked: data missing |
| fig4b-psd-target-bands | not reproduced |
| fig4b-psd-other-bands-unchanged | not reproduced |
| fig5b-iis-post-seizure | reproduced (rests on the seizure end assumption) |
| fig5c-interictal-iis-no-difference | blocked: data missing |
| fig5d-iis-seizure-correlation | blocked: data missing |
| fig6cd-interictal-hfo-no-difference | blocked: data missing |
| fig6e-ictal-ripples-no-difference | reproduced |
| fig6f-ictal-fast-ripples-reduced | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The dandiset gives no seizure onset or end, so both
are found here from the LFP with a line-length rule of this attempt; every duration, GC, PSD, IIS
and HFO result rests on it, and thresholds 1.5 and 3 are shown beside the main 2. The stimulation
times are found from the stimulation artifact. GC is conditional time-domain GC from OLS VAR fits
written here, not the MVGC toolbox. HFOs are automatic detections without the paper's manual
review, with a threshold of this attempt. IIS thresholds use the 180 s before each seizure, as
the paper's interictal segments are not in the dandiset. Target labels are those of Figure S2.

**What was found about the dandiset.**
- Each file holds 600 s epochs around seizures, with no seizure times, stimulation times,
  absolute start times or recording durations, and no interictal segments: five claims cannot be
  tested.
- The order of the stimulation groups differs from Figure S2 for R5 (CA1, ANT, DG, CA3 against
  ANT, DG, CA1, CA3) and R6 (ANT, DG, CA1, CA3 against CA1, ANT, DG, CA3); the epoch dates follow
  the dandiset's order.
- Stimulation trains are found in most seizures of most stimulation groups, but in 1, 1, 0 and 0
  seizures of R7's four groups and in none of R3's CA1 group (R3-07), whose epochs are dated within
  the R3 DG group's days and before R3's last baseline group.
- Four baseline epochs carry trains at the pulse rate of the rat's own stimulator (R1-04 twice,
  R4-00, R6-02).
- Epoch dates of some groups overlap those of the next group (for example R1 2023-09-20 in both
  base3 and DG stimulation), and the date format varies (R4 uses `2023_9-11`). No epoch or channel
  is duplicated.